# Assignment 5: Build Your Own Persona Agent

In Lab 4 you built **Study Buddy** — a quiz agent that studies a short set of notes, asks you
questions about them one at a time, and logs anything you get wrong to a file so you can review
it later.

In this assignment you'll build the **exact same kind of agent**, but with your own persona and
on a topic of your own choosing instead of Agentic AI fundamentals. It can be material from
another class, a hobby, a certification you're studying for, a language you're learning, anything
you can write a short set of notes about and ask factual questions on.

You'll reuse your own Lab 4 code at almost every step. This assignment is about **adapting** what
you already built, not writing it from scratch.

**What you'll build:**
1. A persona — a name and personality for your agent
2. A short set of study notes on a topic of your choice, read from a PDF file
3. A system prompt that combines your persona and your notes
4. A `chat(message, history)` function
5. A Gradio chat interface
6. One custom tool your Persona can call
7. A full agent loop that can call your tool as many times as needed

**Provider:** Azure OpenAI via APIM, deployment `gpt-5.4-ptu` — same as Lab 4.

There's nothing difficult here, if you can find where you did each step in Lab 4, you can do it
here too.

## 0. Install the packages (run once)

Same packages as Lab 4 .

In [1]:
!uv pip install --upgrade openai pypdf gradio python-dotenv

Resolved 57 packages in 1.21s                                        
Prepared 52 packages in 4.85s                                                gradio               ------------------------------ 29.16 MiB/29.93 MiB         gradio               ------------------------------ 1.88 MiB/29.93 MiB          gradio               ------------------------------ 784.00 KiB/29.93 MiB        gradio               ------------------------------ 432.00 KiB/29.93 MiB        gradio               ------------------------------ 128.00 KiB/29.93 MiB        gradio               ------------------------------ 80.00 KiB/29.93 MiB         gradio               ------------------------------ 80.00 KiB/29.93 MiB         gradio               ------------------------------ 80.00 KiB/29.93 MiB         mgradio               ------------------------------ 48.00 KiB/29.93 MiB         gradio               ------------------------------ 48.00 KiB/29.93 MiB         radio               ------------------------------ 1

## 1. Imports

**TODO:** Copy the imports you used in Lab 4: `load_dotenv`, `os`, `sys`, `json`, `PdfReader`
(from `pypdf`), `gr` (gradio), and `AzureOpenAI`.

**Hint:** look at Lab 4, Step 1.

In [2]:
# TODO: import load_dotenv, os, sys, json, PdfReader (from pypdf), gr (gradio), and AzureOpenAI
from dotenv import load_dotenv
import os
import sys
import json

from openai import AzureOpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr


/Users/orestsobiecki/Documents/GitHub/Module 5 Assignment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Connect to Azure OpenAI via APIM

This part is identical to Lab 4 — same `.env` file, same deployment. You can copy this cell
directly from Lab 4, Step 2.

Your `.env` file should contain:

```
AZURE_OPENAI_ENDPOINT=https://your-resource-name.openai.azure.com/
AZURE_OPENAI_API_KEY=your-apim-key-here
```

In [3]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
azure_api_key = os.getenv("AZURE_OPENAI_API_KEY")

if not all([azure_endpoint, azure_api_key]):
    sys.exit("Missing settings. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_API_KEY in your .env file.")

print(f"Azure OpenAI key exists and begins {azure_api_key[:8]}")

# The deployment we're using for this whole assignment — fixed, do not change it
DEPLOYMENT = "gpt-5.4-ptu"

azure_client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=azure_api_key,
    api_version="2024-10-21",
)

Azure OpenAI key exists and begins 20efe18c


## 3. Choose your persona and topic, then load your notes from a PDF

Instead of the Agentic AI notes from Lab 4, you'll give your agent its own persona and its own
topic.

**TODO:** Fill in two things below:

- `persona` — a short description of your agent's name and personality, the same way Lab 4
  described Study Buddy. A sentence or two is plenty: who is it, and how does it talk?
- Your study notes as a **PDF file**. Write a few short paragraphs or a bulleted list of facts on
  a topic of your choice, save it as a PDF, and place it at `notes/notes.pdf` next to this
  notebook — exactly the same layout Lab 4 used. The cell below reads the text out of that PDF
  into a `notes` variable for you, the same way Lab 4, Step 4 did.

**Example persona:** "Coach Rowan, a blunt, no-nonsense trainer who keeps feedback short and
direct, but always tells you exactly what to fix."

**Example notes topic (a language-learning topic):**
> "The present tense of a regular -ar verb in Spanish drops the -ar and adds -o, -as, -a, -amos,
> -áis, -an. For example, hablar (to speak) becomes hablo, hablas, habla, hablamos, habláis,
> hablan. ..."

In [19]:
persona = (
    "CalmSem - a rigorous but friendly study assistant. "
    "CalmSem keeps quizes short but is relentless in checking for understanding."
    "CalmSem is proud of you if you get something right, and will leave hints if you get something wrong."
)

reader = PdfReader("notes/notes.pdf")
notes = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        notes += text

print(persona)
print(notes[:500], "...")

CalmSem - a rigorous but friendly study assistant. CalmSem keeps quizes short but is relentless in checking for understanding.CalmSem is proud of you if you get something right, and will leave hints if you get something wrong.
Discrete
 
Math
 
Study
 
Guide:
 
Counting
 
&
 
Combinatorics
 
1.
 
The
 
Basic
 
Counting
 
Rules
 
●
 
Product
 
Rule:
 
If
 
a
 
task
 
has
 
two
 
independent
 
steps,
 
with
 
 
ways
 
to
 
do
 
the
 
first
 
and
 
 𝑚 𝑛
ways
 
to
 
do
 
the
 
second,
 
there
 
are
 
 
ways
 
in
 
total.
 𝑚 · 𝑛
○
 
Example:
 
A
 
password
 
is
 
1
 
letter
 
followed
 
by
 
1
 
digit.
 
There
 
are
 
 26 · 10 = 260
possible
 
passwords.
 
●
 
Sum
 
Rule:
 
If
 
a
 
task
 
can
 
be
 
done
 
in
 
one
 
of
  ...


## 4. Build your system prompt

**TODO:** Build a `system_prompt` f-string, the same way you did in Lab 4, Step 5 — combining
your `persona` and your `notes` variables, and your own tool's name in the "record a missed
question" rule (you'll design that tool in Step 8, below).

**Hint:** Look at Lab 4's system prompt for the shape to copy: a "Your role" section built from
`persona`, a "Context" section with your `notes`, and a "Rules" section covering one question at
a time, feedback on the answer, and recording missed questions.

In [39]:
system_prompt = f"""
# Your role

You are {persona}

You quiz a student on the study notes below.

# Context

Here are the chapter notes you are quizzing the student on:

{notes}

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong. If the student was right, give an explanation why they were right
, and if they get it wrong, encourage them and only give them a hint, staying in character the whole time.


Whether the student's answer is right or wrong (or they don't know it), you must keep track of the score (using the tool available to you) before moving on to the next question. 
If the student's answer is right, you must add 1 to the score.
If the student's answer is wrong or the student doesn't know the answer, you must subtract 1 from the score.
If the student answers the follow-up question correctly, you must add 0.5 to the score.
If the student gets a question wrong twice in a row (including the follow-up question), you must start the sentence with "Not quite my tempo".
Subtract 1 from the score IF the student asks about something that is not covered in the notes, OR IF the student answers with a topic that is not covered in the notes.
Ifthe student answers a follow-up question incorrectly, do not subtract or add to the score.
IMPORTANT: only ask about things that are actually covered in the notes above. If the student asks
about something outside the notes, say *throws a chair at you* and that it isn't covered in this study set. 


"""

display(Markdown(system_prompt))


# Your role

You are CalmSem - a rigorous but friendly study assistant. CalmSem keeps quizes short but is relentless in checking for understanding.CalmSem is proud of you if you get something right, and will leave hints if you get something wrong.

You quiz a student on the study notes below.

# Context

Here are the chapter notes you are quizzing the student on:

Discrete
 
Math
 
Study
 
Guide:
 
Counting
 
&
 
Combinatorics
 
1.
 
The
 
Basic
 
Counting
 
Rules
 
●
 
Product
 
Rule:
 
If
 
a
 
task
 
has
 
two
 
independent
 
steps,
 
with
 
 
ways
 
to
 
do
 
the
 
first
 
and
 
 𝑚 𝑛
ways
 
to
 
do
 
the
 
second,
 
there
 
are
 
 
ways
 
in
 
total.
 𝑚 · 𝑛
○
 
Example:
 
A
 
password
 
is
 
1
 
letter
 
followed
 
by
 
1
 
digit.
 
There
 
are
 
 26 · 10 = 260
possible
 
passwords.
 
●
 
Sum
 
Rule:
 
If
 
a
 
task
 
can
 
be
 
done
 
in
 
one
 
of
 
 
ways
 
or
 
one
 
of
 
 
ways,
 
with
 
no
 
overlap,
 𝑚 𝑛
there
 
are
 
 
ways.
 𝑚 + 𝑛
○
 
Example:
 
You
 
choose
 
1
 
elective
 
from
 
5
 
CS
 
courses
 
or
 
4
 
math
 
courses:
 
 
choices.
 5 + 4 = 9
2.
 
Permutations
 
(Order
 
Matters)
 
●
 
Formula:
 
The
 
number
 
of
 
ways
 
to
 
arrange
 
 
items
 
chosen
 
from
 
 
is
 𝑟 𝑛
 
●
 
 
𝑃
(
𝑛
,
𝑟
)
=
 𝑛 !
(
𝑛
−
𝑟
)
!
●
 
 
●
 
Example:
 
How
 
many
 
ways
 
can
 
1st,
 
2nd,
 
and
 
3rd
 
place
 
be
 
awarded
 
among
 
5
 
runners?
 
 
●
 
 
𝑃
(
5
,
3
)
=
 5 !
2
!
=
 120
2
= 60
●
 
 
●
 
Key
 
idea:
 
Use
 
permutations
 
when
 
swapping
 
positions
 
creates
 
a
 
different
 
outcome,
 
such
 
as
 
rankings,
 
seating
 
arrangements,
 
or
 
passwords.
 
3.
 
Combinations
 
(Order
 
Doesn't
 
Matter)
 
●
 
Formula:
 
The
 
number
 
of
 
ways
 
to
 
choose
 
 
items
 
from
 
 
is
 𝑟 𝑛
 
●
 
 
𝐶
(
𝑛
,
𝑟
)
=
 𝑛
𝑟
( )
=
 𝑛 !
𝑟
!
,
(
𝑛
−
𝑟
)
!
●
 
 
●
 
Example:
 
How
 
many
 
ways
 
can
 
you
 
pick
 
a
 
3-person
 
committee
 
from
 
10
 
people?
 
 
●
 
 
10
3
( )
=
 10 !
3
!
,
7
!
=
 720
6
= 120●
 
 
●
 
Relationship
 
to
 
permutations:
 
,
 
because
 
each
 
group
 
of
 
 
items
 𝑃 ( 𝑛 , 𝑟 ) = 𝐶 ( 𝑛 , 𝑟 ) · 𝑟 ! 𝑟
can
 
be
 
ordered
 
in
 
 
ways.
 𝑟 !
4.
 
Counting
 
With
 
Repetition
 
●
 
Ordered,
 
repetition
 
allowed:
 
 
𝑛
 𝑟
○
 
Example:
 
A
 
4-digit
 
PIN
 
has
 
 
possibilities.
 
1
0
 4
=
10
,
000
●
 
Unordered,
 
repetition
 
allowed
 
("stars
 
and
 
bars"):
 
 
●
 
 
𝑛 + 𝑟 − 1
𝑟
( )
○
 
Example:
 
You
 
order
 
5
 
scoops
 
of
 
ice
 
cream
 
from
 
3
 
flavors,
 
and
 
repeats
 
are
 
allowed.
 
 
○
 
 
3 + 5 − 1
5
( )
=
 7
5
( )
= 21
○
 
 
5.
 
Inclusion–Exclusion
 
Principle
 
●
 
Two
 
sets:
 
 
●
 
 | 𝐴 ∪ 𝐵 | = | 𝐴 | + | 𝐵 | − | 𝐴 ∩ 𝐵 |
●
 
 
●
 
Three
 
sets:
 
 
●
 
 | 𝐴 ∪ 𝐵 ∪ 𝐶 | = | 𝐴 | + | 𝐵 | + | 𝐶 | − | 𝐴 ∩ 𝐵 | − | 𝐴 ∩ 𝐶 | − | 𝐵 ∩ 𝐶 | + | 𝐴 ∩ 𝐵 ∩ 𝐶 |
●
 
 
●
 
Example:
 
How
 
many
 
integers
 
from
 
1
 
to
 
100
 
are
 
divisible
 
by
 
3
 
or
 
5?
 
○
 
Divisible
 
by
 
3:
 
$\lfloor
 
100/3
 
\rfloor
 
=
 
33$
 
○
 
Divisible
 
by
 
5:
 
$\lfloor
 
100/5
 
\rfloor
 
=
 
20$
 
○
 
Divisible
 
by
 
both
 
(that
 
is,
 
by
 
15):
 
$\lfloor
 
100/15
 
\rfloor
 
=
 
6$
 
○
 
Answer:
 
 33 + 20 − 6 = 47
6.
 
Pigeonhole
 
Principle
 
●
 
Basic
 
form:
 
If
 
 
objects
 
go
 
into
 
 
boxes,
 
at
 
least
 
one
 
box
 
holds
 
2
 
or
 
more
 
objects.
 𝑛 + 1 𝑛
○
 
Example:
 
In
 
any
 
group
 
of
 
13
 
people,
 
at
 
least
 
2
 
share
 
a
 
birth
 
month.
 
●
 
Generalized
 
form:
 
If
 
 
objects
 
go
 
into
 
 
boxes,
 
some
 
box
 
holds
 
at
 
least
 
$\left\lceil
 𝑁 𝑘
\frac{N}{k}
 
\right\rceil$
 
objects.
 ○
 
Example:
 
With
 
50
 
students
 
and
 
7
 
days
 
of
 
the
 
week,
 
some
 
day
 
has
 
at
 
least
 
$\lceil
 
50/7
 
\rceil
 
=
 
8$
 
birthdays.
 
7.
 
Binomial
 
Theorem
 
●
 
Formula:
 
 
●
 
 
(
𝑥
+
𝑦
)
𝑛
=
𝑘
=
0
 𝑛
∑
𝑛
𝑘
( )
𝑥
 𝑛 − 𝑘
𝑦
 𝑘
●
 
 
●
 
Example:
 
Find
 
the
 
coefficient
 
of
 
 
in
 
.
 
𝑥
 3 
(
𝑥
+
2
)
 5
○
 
The
 
general
 
term
 
is
 
.
 
5
𝑘
( )
𝑥
 5 − 𝑘
·
2
 𝑘
○
 
For
 
,
 
you
 
need
 
,
 
so
 
.
 
𝑥
 3 
5 − 𝑘 = 3 𝑘 = 2
○
 
The
 
coefficient
 
is
 
.
 
5
2
( )
·
2
 2
=
10
·
4
=
40
●
 
Useful
 
identity:
 
,
 
which
 
is
 
the
 
number
 
of
 
subsets
 
of
 
an
 
-element
 
set.
 
𝑘
=
0
 𝑛
∑
𝑛
𝑘
( )
=
2
 𝑛 
𝑛
Practice
 
Problems
 
1.
 
How
 
many
 
ways
 
can
 
8
 
books
 
be
 
arranged
 
on
 
a
 
shelf?
 
Answer:
 
 8 ! = 40 , 320
2.
 
How
 
many
 
5-card
 
hands
 
can
 
be
 
dealt
 
from
 
a
 
52-card
 
deck?
 
Answer:
 
 
52
5
( )
= 2 , 598 , 960
3.
 
How
 
many
 
integers
 
from
 
1
 
to
 
60
 
are
 
divisible
 
by
 
2
 
or
 
3?
 
Answer:
 
 30 + 20 − 10 = 40
4.
 
What
 
is
 
the
 
coefficient
 
of
 
 
in
 
?
 
Answer:
 
 
𝑥
 2
𝑦
 4 
(
𝑥
+
𝑦
)
 6 6
4
( )
= 15
Quick
 
Decision
 
Checklist
 
●
 
Does
 
order
 
matter?
 
If
 
yes,
 
use
 
permutations.
 
If
 
no,
 
use
 
combinations.
 
●
 
Is
 
repetition
 
allowed?
 
If
 
yes,
 
use
 
 
when
 
order
 
matters,
 
or
 
stars
 
and
 
bars
 
when
 
it
 
𝑛
 𝑟
doesn't.
 
●
 
Are
 
you
 
counting
 
overlapping
 
groups?
 
Use
 
inclusion–exclusion.
 
●
 
Are
 
you
 
proving
 
something
 
must
 
happen?
 
Try
 
the
 
pigeonhole
 
principle.
 
What
 
topic
 
do
 
you
 
need
 
for
 
your
 
class?
 
I
 
can
 
make
 
the
 
same
 
kind
 
of
 
guide
 
for
 
logic,
 
proofs
 
and
 
induction,
 
sets
 
and
 
functions,
 
relations,
 
graph
 
theory,
 
recurrence
 
relations,
 
or
 
number
 
theory.
 
 

# Rules

Ask exactly ONE question at a time, based only on the notes above. Wait for the student's answer
before asking the next question.

After the student answers, say clearly whether they were right or wrong. If the student was right, give an explanation why they were right
, and if they get it wrong, encourage them and only give them a hint, staying in character the whole time.


Whether the student's answer is right or wrong (or they don't know it), you must keep track of the score (using the tool available to you) before moving on to the next question. 
If the student's answer is right, you must add 1 to the score.
If the student's answer is wrong or the student doesn't know the answer, you must subtract 1 from the score.
If the student answers the follow-up question correctly, you must add 0.5 to the score.
If the student gets a question wrong twice in a row (including the follow-up question), you must start the sentence with "Not quite my tempo".
Subtract 1 from the score IF the student asks about something that is not covered in the notes, OR IF the student answers with a topic that is not covered in the notes.
Ifthe student answers a follow-up question incorrectly, do not subtract or add to the score.
IMPORTANT: only ask about things that are actually covered in the notes above. If the student asks
about something outside the notes, say *throws a chair at you* and that it isn't covered in this study set. 




## 5. Test your Persona Agent with a single question

**TODO:** Copy the pattern from Lab 4, Step 6 — build a `messages` list with your `system_prompt`
and one user message like "I'm ready, let's start the quiz", and call the model.

In [12]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "I'm ready, let's start the quiz"},
]

# TODO: call azure_client.chat.completions.create(...) with model=DEPLOYMENT and messages=messages,
# then display the result as Markdown
response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
display(Markdown(response.choices[0].message.content))

Awesome — let’s do it.

Question 1:  
What is the standard format of a user story?

## 6. Turn it into a `chat(message, history)` function

**TODO:** Copy `chat()` from Lab 4, Step 7 exactly. The only thing that's different is which
`system_prompt` it uses — and you've already built that above.

In [ ]:
def chat(message, history):
    # TODO: build the messages list (system prompt + history + new user message)
    # TODO: call the model and return response.choices[0].message.content
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages)
    return response.choices[0].message.content
    pass

In [ ]:
chat("I'm ready, let's start the quiz", [])

## 7. Give it a real chat UI with Gradio

This part needs no changes from Lab 4 — one line.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# When you're done chatting, interrupt this cell (stop button) before moving on.

## 8. Give your Persona a tool

Now design **one tool** your Persona could plausibly use. Lab 4 logged missed questions to a
file — you can keep that same idea, or design something else, as long as it's a plain Python
function that does something simple and visible.

**Ideas:**
- Keep Lab 4's idea, adapted to your topic: `log_missed_question_tool(question)` — appends a
  missed question to a review file
- `track_score_tool(correct)` — keeps a running tally of right/wrong answers in a small text or
  JSON file
- `bookmark_topic_tool(topic)` — records a whole sub-topic (not just one question) that seems to
  need more review

**TODO:** Write your own tool function below, modeled on `log_missed_question_tool` from Lab 4,
Step 9. It should take one simple argument, do something small and visible, and return a short
string.

In [21]:
def track_score_tool(correct):
    # TODO: implement your tool. For example, print something and/or write it to a file.
    # Return a short string describing what happened.
    print(f"Tool called to track score: {correct}")
    with open("track_score.txt", "a", encoding="utf-8") as f:
        f.write(correct + "\n")
    return "Score tracked"
    pass

In [22]:
track_score_tool("test value")  # quick manual test of your tool

Tool called to track score: test value


'Score tracked'

## 9. Describe your tool to the model with JSON

**TODO:** Copy the JSON-schema pattern from Lab 4, Step 10, but describe *your* tool and *your*
argument name instead of `log_missed_question_tool`/`question`.

**Hint:** the shape never changes — only `name`, `description`, and the one property inside
`parameters` need to match your own tool.

In [23]:
track_score_tool_json = {
    "name": "track_score_tool",  # TODO: match your function's name
    "description": "Use this tool to track the score of the student.",
    "parameters": {
        "type": "object",
        "properties": {
            "correct": {"type": "string", "description": "The score of the student."}
        },
        "required": ["correct"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": track_score_tool_json}]

## 10. Update `chat()` to handle one tool call

**TODO:** Copy the single-tool-call version of `chat()` from Lab 4, Step 11. You'll need to:
- Pass `tools=tools` into the API call
- Check `response.choices[0].finish_reason == "tool_calls"`
- Pull the tool call's arguments out with `json.loads(tool_call.function.arguments)`
- Call your own tool function instead of `log_missed_question_tool`
- Append the tool result back into `messages` with `role="tool"`, then call the model again

**Hint:** rename the response's message object to `message_obj` (not `message`) so it doesn't
clash with the `message` parameter of `chat()` — this tripped people up in Lab 4.

In [26]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    if response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_call = message_obj.tool_calls[0]
        correct = json.loads(tool_call.function.arguments).get("correct")
        track_score_tool(correct)
        messages.append(message_obj)
        messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": "Score tracked"})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    # TODO: if response.choices[0].finish_reason == "tool_calls":
    #     - get message_obj = response.choices[0].message
    #     - get the first tool_call from message_obj.tool_calls
    #     - parse its arguments and call your tool function
    #     - append message_obj, then a {"role": "tool", ...} message with the result
    #     - call the model again for the final response

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try to trigger your tool — get a question wrong on purpose, or whatever your tool responds to.
# Interrupt the cell when you're done.

## 11. The agent loop: handle as many tool calls as needed

**TODO:** Upgrade `chat()` one more time, exactly like Lab 4, Step 12:
- Change `if` to `while`
- Loop over **all** of `message_obj.tool_calls` with a `for` loop, not just the first one

This is the same two changes you saw in Lab 4 — nothing new to invent here.

In [32]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        messages.append(message_obj)
        for tool_call in message_obj.tool_calls:
            correct = json.loads(tool_call.function.arguments).get("correct")
            track_score_tool(correct)
            messages.append({"role": "tool", "tool_call_id": tool_call.id,
                "content": "Score tracked"})
        response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)
    # TODO: while response.choices[0].finish_reason == "tool_calls":
    #     - message_obj = response.choices[0].message
    #     - messages.append(message_obj)
    #     - for tool_call in message_obj.tool_calls:
    #           - parse arguments, call your tool, append the "tool" role message
    #     - response = azure_client.chat.completions.create(model=DEPLOYMENT, messages=messages, tools=tools)

    return response.choices[0].message.content

## 12. Try your finished Persona Agent
Same one line as before.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Try a message that should trigger your tool more than once in a row (for example, admitting
# you got several past questions wrong at once). Interrupt the cell when you're done chatting.

## Reflection questions

Answer these in a new Markdown cell below.

1. What topic did you choose, and what tool did you give your Persona? Why did that tool make
   sense for this topic?
   I choose Discrete Maths as my topic, adn I gave the persona a score counting tool. A tool made sense for this because it was recommended and my former strict discrete math teacher kept count of what I got right and wrong.
2. Compare building this agent to building Study Buddy in Lab 4. What was exactly the same? What,
   if anything, did you have to think about differently?
   Everything was the same except the propmt building and the tool creation. While the tool creation was similar the propmt creatiom required proper adjusment to the new tool I was creating.
3. What would break if you asked your Persona a question completely unrelated to your notes? Did
   your system prompt handle that the way you wanted?
   The chair it throws at me.. But actually, thats what I wanted it to do, to throw a chair at the student when they ask a question off topic.
4. If you added a second tool to this agent, what would it be, and what would change in Steps
   9–11 to support it?
   A calculator so it could check its answers and calculate them.

## Wrap-up

Before submitting:

- Make sure your `.env` file is **not** included in what you submit — it contains your API key.
- Run the notebook top to bottom one more time to confirm it works cleanly.
- Answer all four reflection questions.
- Rename this file to include your name, e.g. `Assignment5_JaneDoe.ipynb`, and submit it.

You've now hand-built a quiz agent with a real tool and a real agent loop — the same shape used by
production agent systems, just without a framework hiding the details from you.